# 🩺 Exercise 7: Clinical AI in Dermatology – Evaluating the "Black Box"

Welcome to Exercise 7! Today, we are shifting our focus from writing code to **evaluating clinical AI**. As future medical professionals, you will rarely need to build a neural network from scratch. However, you *will* need to understand how to train them, how to evaluate their safety, and how to interpret their decisions.

In this interactive clinical case study, we will explore:
1. **The Dermatology Dataset:** Applying a Convolutional Neural Network (CNN) to diagnose real-world dermatoscopic images.
2. **Data Splitting:** Organizing patient data to ensure the AI learns safely without "memorizing" the answers.
3. **Model Tuning:** Acting as the "attending physician" to adjust how the AI learns.
4. **The Confusion Matrix:** Analyzing the model's behavior, specifically looking at false positives and false negatives.

---

### Core AI Concepts

Before we begin, let's briefly visualize three core concepts you will be controlling today.

#### 1. Data Splitting:
To train an AI, we can't just show it all our data at once. We have to split the patient images into three distinct groups:

| 📚 Training Set (70%) | 📝 Validation Set (10%) | 🎓 Test Set (20%) |
| :--- | :--- | :--- |
| **The learning** | **The Sanity Check** | **Performance on never seen, never sanity checked data** |
| The AI studies these images over and over to learn the patterns of different skin lesions. | The AI is challenged on new images not included during training to see if it's actually learning or just memorizing. | Completely unseen patients. We use this at the very end to prove the AI is safe for the real world. |

#### 2. The Learning Rate
Remember that most, if not all, neural networks are trained by penalizing the network if it does mistakes. The **Learning Rate** dictates how large of a "step" the AI takes when correcting its mistakes. If we assume the loss landscape (the mistake landscape) is a valley, we try to find its lowest point. You should remember from the lectures that we utilize the gradients via backpropagation for this learning procedure. The learning rate just determins the step size towards possible low points in this complex loss landscape.

#### 3. What is an "Epoch"?
An **Epoch** is simply one complete pass through the entire Training Set. 

🔄 **The Epoch Loop:**
1. AI looks at *all* the training images.
2. AI makes predictions.
3. AI checks its answers and updates its "brain" (adjusts its weights).
4. **End of 1 Epoch.** *(If we set `epochs = 3`, the AI will read the training data three times from start to end to reinforce its learning).*

---

### Step 1: Data Loading

In previous exercises, you might remember the tedious process of loading image files manually, analyzing them pixel by pixel. In a high-volume clinical pipeline, that is far too slow and prone to memory crashes. 

Today, we are automating this process using **PyTorch Datasets and DataLoaders**. These modules automatically fetches the patient files, shuffles them to prevent the AI from memorizing the order, groups them into **Batches**, and hands them to the model.

When we run the cell below, our automated coordinator returns five distinct objects:
* **`train_loader`, `val_loader`, `test_loader`:** The automated dispensers that will feed batches of images to our AI during the different phases of training and testing.
* **`labels`:** Our diagnostic dictionary, mapping the AI's numerical output (e.g., `4`) to the human-readable diagnosis (e.g., "Melanoma").
* **`train_dataset`:** The raw collection of training records, which we will use later to analyze the prevalence of different diseases.

*(**For the curious:** We have hidden the engineering code that downloads and formats these images to keep your dashboard clean. If you want to see exactly how to write a PyTorch DataLoader from scratch, open the `utils.py` file in your file explorer!)*

### 

In [ ]:
!pip install medmnist torchsummary --quiet

# Run this cell to load our tools; an interested reader may want to peek into the code to see how it works!
# to do so, open the file utils.py in this workspace.
import utils
import numpy as np


print("Tools loaded successfully.")

In [ ]:
# visualize one image per class
utils.plot_sample_images(labels)

In [ ]:
# Load the dataset
train_loader, val_loader, test_loader, labels, train_dataset, val_dataset, test_dataset = utils.load_dermamnist(batch_size=128)

print("\n--- Diagnostic Categories ---")
for key, value in labels.items():
    print(f"Class {key}: {value}")

train_samples_per_class = utils.get_number_of_samples_per_class_as_table(train_dataset, labels)
val_samples_per_class = utils.get_number_of_samples_per_class_as_table(val_dataset, labels)
test_samples_per_class = utils.get_number_of_samples_per_class_as_table(test_dataset, labels)

### Step 2: Understanding Data Splits

We have successfully initialized our automated DataLoaders! They will handle the heavy lifting of looping through the patient data during the training, validation, and testing phases. 

But before we unleash the AI to classify these skin lesions, let's pause and think critically about *how* we evaluate success. 

> #### Group Discussion: Which AI is Safer?
> Imagine you are evaluating two different AI models for your dermatology clinic:
> 
> * **Model A:** Achieves a **98%** accuracy on the *Training Set*, but only **80%** accuracy on the *Validation Set*.
> * **Model B:** Achieves an **87%** accuracy on the *Training Set*, and an **85%** accuracy on the *Validation Set*.
> 
> **Discuss with your neighbor (3 minutes):**
> 1. Which model would you choose to deploy in your clinic and why?
> 2. Is one model objectively "better" than the other? Why or why not?

Please answer these questions before continuing below.

In [ ]:
# -------------------------------------------------------------------------
# EXPERIMENT: Small Train Set
# -------------------------------------------------------------------------
# Let's test the theory from our discussion. We will train a fresh AI 
# on a VERY small subset of patients (only 50 images) for many epochs. .

small_train_loader, small_train_dataset = utils.get_small_train_loader(train_dataset, num_samples=50)

# Build a fresh AI Model
model = utils.get_model()

print("🚨 Starting Overfitting Experiment...")
print("Watch how the Train Accuracy climbs near 100%, but Validation Accuracy stays poor!\n")

# 3. Train for 15 epochs to force memorization
# Notice we test it against the FULL validation set to expose the memorization!
trained_overfit_model = utils.train_model(
    model, 
    small_train_loader, 
    val_loader, 
    small_train_dataset,
    epochs=100, 
    learning_rate=0.001, 
    use_class_weights=False
)

#### 💡 The Importance of Generalization
We can see that it is trivial to achieve near-perfect accuracy on the training data; modern neural networks possess enough capacity to completely memorize the dataset. 

Because of this, training accuracy is a poor metric for clinical safety. When performing model selection, we must always rely on a strictly quarantined **Validation Dataset**. The model never learns from these images. Instead, this hold-out data serves as an objective benchmark to verify whether the AI has learned true diagnostic patterns that will **generalize** to unseen data, rather than just memorizing the past.

---

### Step 3: Training on Full Data

Now that we understand the dangers of overfitting to a small group of patients, let's open the doors to the full dataset. We are going to train a fresh CNN model on the **entire training dataset**.

As a clinical decision maker, you must perform **Model Selection and Tuning**. You will adjust two key parameters:
1. **Learning Rate:** How aggressively the AI updates its medical knowledge after making a mistake.
2. **Epochs:** How many times the model reviews the complete set of training patient files.

> **🎯 Your Mission:** Play around with different combinations of Learning Rates and Epochs using the codeblock below. Your goal is to find the model that achieves the **highest Validation Accuracy**. 
>
> *Keep an eye out for overfitting! If your Training Accuracy keeps climbing but your Validation Accuracy stalls or drops, the model has stopped learning and started memorizing.*

In [ ]:
# Get a fresh AI model
new_model = utils.get_model()

learning_rate = 0.001 # choose a learning rate, for example 0.01, 0.001, 0.0001 (or any other small number in between and beyond) You can experiment with this value.
epochs = 10 # choose a number of epochs, for example 1, 5, 10,... Start small and see how it goes!

print("Training Model... please wait.")
trained_model = utils.train_model(
    new_model, 
    train_loader, 
    val_loader, 
    train_dataset,
    epochs=epochs, 
    learning_rate=learning_rate, 
    use_class_weights=False  # We will look at this soon!
)

### Step 4: Performance Analysis and Class Imbalance

In the previous steps, we relied on overall validation accuracy to guide our model selection. However, a critical characteristic of real-world medical datasets was temporarily abstracted away: the prevalence of different conditions is rarely uniform. 

Before training any model, it is imperative to analyze the class distribution within your dataset. Failing to do so can lead to highly misleading performance metrics. Let us examine the distribution of our classes in the training and validation data. (note: in our case, the relative proportions are the same across splits)

### Count Training Samples per Class

In [ ]:
train_samples_per_class

### Count Validation Samples per Class

In [ ]:
val_samples_per_class

> #### 📊 Analytical Exercise
> Review the distribution printed above. 
> 
> 1. Given the highly skewed class distributions across the dataset splits, do you see any fundamental problems that might arise during training?
> 2. How might the network's internal optimization (loss minimization) implicitly favor certain classes over others?
> 
> *Discuss these implications briefly before proceeding.*

#### The Fallacy of Global Accuracy

When dealing with highly imbalanced datasets, simple global validation accuracy is often an uninformative metric. To illustrate this, let us consider a "naive classifier": a rigid algorithm that completely ignores the input image and simply outputs the majority class ("melanocytic nevi") for every single prediction.

> #### 🧮 Calculation Task
> 1. Given the sample counts per class in the validation set, calculate the expected global accuracy of this naive classifier that always predicts melanocytic nevi (do this on paper or in a calculator, no code required). (we provide the answer to this question in the upcoming code section. But please try doing the calculation first on your own or in a group. Below is a hint.)
> 
> 💡 **Hint:** *Imagine you have 100 patients in a waiting room. If 70 of them have a common nevus, and you blindly diagnose every single person in the room with a nevus without even looking at them, how many times would you be correct? How would you calculate that percentage using the real patient numbers from our validation dataset?*
> 
> 2. Compare this baseline percentage to the validation accuracy your trained model achieved in Step 3.

It should become immediately evident that a completely non-intelligent classifier can achieve remarkably high global accuracy simply by exploiting statistical prevalence. Consequently, achieving around 70% accuracy on this dataset is not an indicator of a clinically useful model. 

To properly assess our model, we must compute its **class-conditional performance**, particularly on the minority classes (such as Class 3). We also show the performance on the majority class, see below. What do you observe?

In [ ]:
import numpy as np

# 1. Calculate the accuracy of the Naive Classifier on the validation set
val_labels = val_dataset.labels.squeeze()
val_classes, val_counts = np.unique(val_labels, return_counts=True)

majority_class_idx = np.argmax(val_counts)
majority_class_count = val_counts[majority_class_idx]
total_val_samples = len(val_labels)

naive_accuracy = majority_class_count / total_val_samples

print(f"Majority Class: {labels[str(majority_class_idx)]}")
print(f"Naive Classifier Accuracy (predicting only majority): {naive_accuracy * 100:.2f}%\n")

# 2. Evaluate our trained model specifically on the minority class (Class 3)
print("Evaluating our trained model on the minority pathology...")
utils.evaluate_specific_class(
    model=trained_model, 
    data_loader=val_loader, 
    target_class_idx=3, 
    label_dict=labels
)

print("Evaluating our trained model on the majority pathology...")
utils.evaluate_specific_class(
    model=trained_model, 
    data_loader=val_loader, 
    target_class_idx=5, 
    label_dict=labels
)

#### Comprehensive Performance Estimation: The Confusion Matrix

As established in your prior exercises, relying on a single scalar metric is insufficient for rigorous model selection. We require a deeper analysis of the error distribution.

Instead of doing the class-wise accuracy for every class and print them, we can build the well-known and established confusion matrix.

The **Confusion Matrix** provides a holistic view of the model's behavior, allowing us to simultaneously evaluate true positives, false positives, true negatives, and false negatives across **all classes**. 

Run the cell below to generate the matrix for your model. Observe where the highest concentrations of misclassifications occur. Inspecting the confusion matrix, can you confirm the pattern we observed above? What patterns do you perceive?

In [ ]:
# Evaluate the AI on the test patients

utils.evaluate_and_plot_confusion_matrix(trained_model, test_loader, labels)

### Step 5: Mitigating Imbalance with Weighted Loss Functions

As observed in our performance analysis, standard training disproportionately favors the majority class. This occurs because the default optimization objective (the loss function) treats all errors equally. In a highly imbalanced dataset, the network can easily minimize its overall loss by simply defaulting to the most frequent pathology.

To counteract this statistical bias, we can introduce **class weights** into our loss calculation (specifically, Weighted Cross-Entropy Loss). By assigning a higher mathematical penalty to misclassifications of minority classes, typically inversely proportional to their frequency in the training data, we force the optimizer to prioritize learning the features of rare conditions.

In this final experiment, we will retrain our network using median frequency balancing. Run the cell below and carefully observe how this mathematical penalty shifts the distribution of errors in the resulting confusion matrix, particularly for our minority classes.

We have to set: use_class_weights=True for this case. Again, we note that we have hidden the complexity of this method into the utils.py file. An interested reader can find the implementation details there.

> #### 🔬 Post-Experiment Analysis
> Examine your new confusion matrix and compare it carefully to the unweighted matrix from Step 3. What patterns do you see now?

In [ ]:
# Get a fresh model
balanced_model = utils.get_model()

epochs = 100 # hold this fixed for now
learning_rate = 0.0001 # hold this fixed for now

print("Retraining with Class Weights ON to prioritize rare diseases...")
balanced_model = utils.train_model(
    balanced_model, 
    train_loader, 
    val_loader, 
    train_dataset,
    epochs=epochs,
    learning_rate=learning_rate,
    use_class_weights=True  # <--- THIS IS THE MAGIC SWITCH!
)

# Let's look at the new confusion matrix
utils.evaluate_and_plot_confusion_matrix(balanced_model, test_loader, labels)

In [ ]:
print("Evaluating our trained model on the minority pathology...")
utils.evaluate_specific_class(
    model=balanced_model, 
    data_loader=val_loader, 
    target_class_idx=3, 
    label_dict=labels
)

print("Evaluating our trained model on the majority pathology...")
utils.evaluate_specific_class(
    model=balanced_model, 
    data_loader=val_loader, 
    target_class_idx=5, 
    label_dict=labels
)

### Step 6: Interpreting the Trade-offs of Weighted Loss

Applying class weights fundamentally alters the model's decision boundaries. By mathematically penalizing errors on rare pathologies, we force the AI to be more cautious and attentive to those specific visual features. However, in machine learning—as in medicine—interventions rarely come without trade-offs.

Typically, in a model that has not been exhaustively fine-tuned, introducing class weights results in a notable increase in accuracy for the minority classes, but at the expense of a performance drop on the majority class. The model becomes more "paranoid" about missing a rare disease.

> #### 🔬 Post-Experiment Analysis
> Examine your new confusion matrix and compare it carefully to the unweighted matrix from Step 3. Discuss the following questions:
> 
> 1. **The Minority Gain:** Did the absolute number of correct predictions for the rarest classes (e.g., Class 3) increase? 
> 2. **The Majority Penalty:** Look at the most common class (Melanocytic Nevi). Did the model's accuracy on this specific class decrease? Are there now more "false positives" where the model suspects a severe pathology when the lesion is actually benign?
> 3. **The Clinical Dilemma:** Suppose your weighted model now successfully flags nearly all true melanomas (high sensitivity) but triggers many false alarms on benign moles (lower specificity). In a real-world dermatological screening pipeline, is this weighted model preferable to the original naive model? Why or why not?